    0. CARGA DE DATOS

In [ ]:
import pandas as pd
import numpy as np

# CARGA DE DATOS
# Cargar el archivo liviano creado en el archivo anterior
    # Recordatorio de primer filtro realizado en dicho archivo:
        # Se filtraron sólo estos códigos de países: ["AR", "BR", "CL", "CO, "MX", "PE"]
df = pd.read_csv('covid_liviano.csv')

# Convertir la columna 'date' a formato fecha real
df['date'] = pd.to_datetime(df['date'])

# Ordenar por País y Fecha
df = df.sort_values(by=['country_name', 'date'])

print(f"Dimensiones: {df.shape}")

Dimensiones: (5946, 50)


    1. OBTENER INFORMACIÓN DEL DATASET

In [90]:
#1.1 Ver el nombre de las columnas
print(df.columns)

Index(['location_key', 'date', 'country_code', 'country_name', 'new_confirmed', 'new_deceased', 'cumulative_confirmed', 'cumulative_deceased', 'cumulative_vaccine_doses_administered', 'population', 'population_male', 'population_female', 'population_rural', 'population_urban', 'population_density', 'human_development_index', 'population_age_00_09', 'population_age_10_19', 'population_age_20_29', 'population_age_30_39', 'population_age_40_49', 'population_age_50_59', 'population_age_60_69', 'population_age_70_79', 'population_age_80_and_older', 'gdp_usd', 'gdp_per_capita_usd', 'latitude', 'longitude', 'area_sq_km', 'smoking_prevalence', 'diabetes_prevalence', 'infant_mortality_rate', 'nurses_per_1000', 'physicians_per_1000', 'average_temperature_celsius', 'minimum_temperature_celsius', 'maximum_temperature_celsius', 'rainfall_mm', 'relative_humidity', 'population_largest_city', 'area_rural_sq_km', 'area_urban_sq_km', 'life_expectancy', 'adult_male_mortality_rate',
       'adult_female_m

In [91]:
#1.2. Ver informaciòn sobre los tipos de datos de las columnas 
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5946 entries, 0 to 5945
Data columns (total 50 columns):
 #   Column                                 Non-Null Count  Dtype         
---  ------                                 --------------  -----         
 0   location_key                           5946 non-null   object        
 1   date                                   5946 non-null   datetime64[ns]
 2   country_code                           5946 non-null   object        
 3   country_name                           5946 non-null   object        
 4   new_confirmed                          5925 non-null   float64       
 5   new_deceased                           5925 non-null   float64       
 6   cumulative_confirmed                   5925 non-null   float64       
 7   cumulative_deceased                    5925 non-null   float64       
 8   cumulative_vaccine_doses_administered  3173 non-null   float64       
 9   population                             5946 non-null   float64 

In [ ]:
#1.3 Ver rango de fechas de datos por país
print("\n--- RANGO POR PAÍS ---")

# Agrupar por 'country_name' y seleccionar la columna 'date'
rango_paises = df.groupby('country_name')['date'].agg(['min', 'max'])

print(rango_paises)


--- RANGO POR PAÍS ---
                    min        max
country_name                      
Argentina    2020-01-01 2022-09-17
Brazil       2020-01-01 2022-09-17
Chile        2020-01-01 2022-09-17
Colombia     2020-01-01 2022-09-17
Mexico       2020-01-01 2022-09-17
Peru         2020-01-01 2022-09-17


    2. OBTENER INFORMACIÓN DE LOS NULOS DEL DATASET

In [ ]:
#2.1. Contar cuántas columnas tienen el 100% de datos nulos (axis=0 para columnas)
cant_cols_nulas = df.isnull().all(axis=0).sum()

print(f"Hay {cant_cols_nulas} columnas con el 100% de sus datos nulos.")

Hay 0 columnas con el 100% de sus datos nulos.


In [ ]:
#2.2 Contar cuántas filas tienen el 100% de datos nulos (axis=0 para filas)
cant_filas_nulas = df.isnull().all(axis=1).sum()

print(f"Hay {cant_filas_nulas} filas con el 100% de sus datos nulos.")

Hay 0 filas con el 100% de sus datos nulos.


In [116]:
#2.3 Ver cantidad de nulos y porcentaje de los mismos por columna que tenga al menos 1 Nan

#Calcular la cantidad de nulos por columna
nulos_cantidad = df.isnull().sum()

#Calcular el porcentaje de nulos por columna
nulos_porcentaje = (df.isnull().mean() * 100).round(2)

#Creación de Data Frame
df_nulos = pd.DataFrame({
    "Cantidad Nulos": nulos_cantidad,
    "Porcentaje (%)": nulos_porcentaje
})

#Filtrar para ver SOLO las columnas que tienen algún nulo (ordenadas de mayor a menor)
df_nulos_filtrado = df_nulos[df_nulos["Cantidad Nulos"] > 0].sort_values("Porcentaje (%)", ascending=False)

print("--- REPORTE DE VALORES FALTANTES ---")
display(df_nulos_filtrado)

--- REPORTE DE VALORES FALTANTES ---


,Cantidad Nulos,Porcentaje (%)
cumulative_recovered,4178,70.27
new_recovered,3345,56.26
cumulative_vaccine_doses_administered,2773,46.64
rainfall_mm,137,2.30
average_temperature_celsius,42,0.71
relative_humidity,42,0.71
maximum_temperature_celsius,41,0.69
minimum_temperature_celsius,41,0.69
cumulative_deceased,21,0.35
cumulative_confirmed,21,0.35


    3. TRATAMIENTO DE LOS NULOS

        3.1. Creación de df_recortado para comenzar a reemplazar los nulos


In [96]:
# 1. Creamos una COPIA INDEPENDIENTE para no tocar el original por error
# Usar .copy() para que sean dos objetos totalmente distintos en memoria
df_recortado = df.copy()

#Asegurar que la columna fecha sea entendida como fecha (y no texto)
df_recortado['date'] = pd.to_datetime(df_recortado['date'])

        3.2. TRATAMIENTO DE COLUMNAS CON DATOS DE ACUMULACIÓN
(cumulativa_deceased y cumulative_confirmed) --> Ambas con sólo 21 datos Nan

In [97]:

#Configuración para que NO oculte ninguna columna ni fila
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000) # Para que use todo el ancho de la pantalla

#Filtrar las 21 filas con datos nulos
filas_nulas_muertes = df[df['cumulative_deceased'].isnull()]

print(f"--- 21 Filas con Nulos en  ---")
print(filas_nulas_muertes)


--- 21 Filas con Nulos en  ---
     location_key       date country_code country_name  new_confirmed  new_deceased  cumulative_confirmed  cumulative_deceased  cumulative_vaccine_doses_administered   population  population_male  population_female  population_rural  population_urban  population_density  human_development_index  population_age_00_09  population_age_10_19  population_age_20_29  population_age_30_39  population_age_40_49  population_age_50_59  population_age_60_69  population_age_70_79  population_age_80_and_older       gdp_usd  gdp_per_capita_usd  latitude  longitude  area_sq_km  smoking_prevalence  diabetes_prevalence  infant_mortality_rate  nurses_per_1000  physicians_per_1000  average_temperature_celsius  minimum_temperature_celsius  maximum_temperature_celsius  rainfall_mm  relative_humidity  population_largest_city  area_rural_sq_km  area_urban_sq_km  life_expectancy  adult_male_mortality_rate  adult_female_mortality_rate  pollution_mortality_rate  comorbidity_mortali

*se observa que las filas con fecha posterior a 13/09/2022 suelen estar vacías, salvo con los datos estructurales del país

            3.2.1 FILTRO DE FECHAS


In [98]:

# Aplciar el filtro (borrar filas luego del 13/09/2022) 
fecha_corte = '2022-09-13'
df_recortado = df_recortado[df_recortado['date'] <= fecha_corte]

# 4. Verificar
print(f"--- REPORTE DE RECORTE ---")
print(f"Fecha máxima en el nuevo dataset: {df_recortado['date'].max().date()}")
print(f"Filas originales: {len(df)}")
print(f"Filas nuevas: {len(df_recortado)}")
print(f"Se eliminaron {len(df) - len(df_recortado)} filas del final.")


--- REPORTE DE RECORTE ---
Fecha máxima en el nuevo dataset: 2022-09-13
Filas originales: 5946
Filas nuevas: 5922
Se eliminaron 24 filas del final.


        3.3. TRATAMIENTO DE COLUMNAS DE CLIMA
('relative_humidity', 'average_temperature_celsius', 'maximum_temperature_celsius', 'minimum_temperature_celsius')--> Con sólo 41/42 datos Nan

In [99]:
# Columnas de clima para modificar
cols_clima = [
    'relative_humidity', 
    'average_temperature_celsius', 
    'maximum_temperature_celsius', 
    'minimum_temperature_celsius'
]

if cols_clima:
    # INTERPOLACIÓN: Rellenar los huecos intermedios trazando una línea entre el día anterior y el siguiente
    # Agrupado por país
    grupo_paises = df_recortado.groupby('country_name')[cols_clima]
    df_recortado[cols_clima] = grupo_paises.transform(lambda x: x.interpolate())

    # Para los datos del primer y último día
    # Usar backfill (copiar el siguiente) y ffill (copiar el anterior) para cubrir esos extremos
    df_recortado[cols_clima] = df_recortado[cols_clima].bfill().ffill()

    # VERIFICACIÓN
    nulos_restantes = df_recortado[cols_clima].isnull().sum().sum()
    if nulos_restantes == 0:
        print("Temperatura y humedad quedaron completas (0 nulos).")
    else:
        print(f"Todavía quedan {nulos_restantes} nulos en estas columnas.")
else:
    print("No encontré columnas de clima en este dataframe.")

Temperatura y humedad quedaron completas (0 nulos).


In [100]:
#CHEQUEO DE NULOS
# Calcular la cantidad de nulos por columna 
nulos_cantidad = df_recortado.isnull().sum()

# Calcular el porcentaje por columna
nulos_porcentaje = (df_recortado.isnull().mean() * 100).round(2)

# Crear el DataFrame de resumen
df_nulos = pd.DataFrame({
    "Cantidad Nulos": nulos_cantidad,
    "Porcentaje (%)": nulos_porcentaje
})

# Filtrar para ver SOLO las columnas que tienen algún nulo
df_nulos_filtrado = df_nulos[df_nulos["Cantidad Nulos"] > 0].sort_values("Porcentaje (%)", ascending=False)

print("--- REPORTE DE VALORES FALTANTES ---")
display(df_nulos_filtrado)

--- REPORTE DE VALORES FALTANTES ---


,Cantidad Nulos,Porcentaje (%)
cumulative_recovered,4156,70.18
new_recovered,3323,56.11
cumulative_vaccine_doses_administered,2751,46.45
rainfall_mm,113,1.91


        3.4. TRATAMIENTO DE COLUMNAS CON DATOS DE ACUMULACIÓN
(cumulative_vaccine_doses_administered) --> tiene gran porcentaje de nulos. 
SE OBSERVA QUE EN EL AÑO 2020 NO EXISTÍAN VACUNAS

In [101]:
# Crear una columna auxiliar para identificar el año
anio_temp = df_recortado['date'].dt.year

# Crear una etiqueta para identificar si el dato es Nulo o Válido
estado_dato = df_recortado['cumulative_vaccine_doses_administered'].isnull().map({
    True: 'Nulos (Sin Vacunas)', 
    False: 'Válidos (Con Vacunas)'
})

# Generar la tabla comparativa contando cuántos hay de cada tipo por año
tabla_comparativa = pd.crosstab(anio_temp, estado_dato)

# Ordenar las columnas para facilitar la lectura
cols_orden = ['Nulos (Sin Vacunas)', 'Válidos (Con Vacunas)']

print("--- COMPARACIÓN DE DATOS DE VACUNAS POR AÑO ---")
display(tabla_comparativa)

# Calcular fecha máxima de nulos
filas_sin_vacunas = df_recortado[df_recortado['cumulative_vaccine_doses_administered'].isnull()]
if not filas_sin_vacunas.empty:
    fecha_max_nulo = filas_sin_vacunas['date'].max()
    print(f"\nEl último día con dato nulo de vacunas es: {fecha_max_nulo.date()}")

--- COMPARACIÓN DE DATOS DE VACUNAS POR AÑO ---


cumulative_vaccine_doses_administered,Nulos (Sin Vacunas),Válidos (Con Vacunas)
date,,
2020,2184,12
2021,226,1964
2022,341,1195



El último día con dato nulo de vacunas es: 2022-09-13


In [102]:
# Filtrar filas donde la columna de vacunas no es nula y es mayor a 0
# Esto asegura tomar el primer registro real de vacunación
filtro_con_vacunas = df_recortado[df_recortado['cumulative_vaccine_doses_administered'] > 0]

# Agrupar por país y buscar la fecha mínima de registro
inicio_registro_vacunacion = filtro_con_vacunas.groupby('country_name')['date'].min().sort_values()

print("--- FECHA DE INICIO DE REGISTRO DE VACUNACIÓN POR PAÍS ---")
display(inicio_registro_vacunacion)

--- FECHA DE INICIO DE REGISTRO DE VACUNACIÓN POR PAÍS ---


country_name
Chile       2020-12-24
Mexico      2020-12-24
Argentina   2020-12-29
Brazil      2021-01-17
Peru        2021-02-09
Colombia    2021-02-17
Name: date, dtype: datetime64[ns]

In [103]:
# Definir diccionario con fechas de inicio de registro de vacunación por país
fechas_inicio_registro_vacunacion = {
    'Chile': '2020-12-24',
    'Mexico': '2020-12-24',
    'Argentina': '2020-12-29',
    'Brazil': '2021-01-17',
    'Peru': '2021-02-09',
    'Colombia': '2021-02-17'
}

# Iterar sobre cada país y su fecha de inicio para rellenar con 0 los periodos previos
for pais, fecha_str in fechas_inicio_registro_vacunacion.items():
    # Asegurar formato datetime para la comparación
    fecha_limite = pd.to_datetime(fecha_str)

    # Identificar filas del país actual con fecha anterior al inicio de vacunación
    condicion = (df_recortado['country_name'] == pais) & (df_recortado['date'] < fecha_limite)

    # Rellenar con 0 solo en esas filas específicas
    df_recortado.loc[condicion, 'cumulative_vaccine_doses_administered'] = \
        df_recortado.loc[condicion, 'cumulative_vaccine_doses_administered'].fillna(0)

# Verificar cuántos nulos quedan después de la limpieza
# Deberían quedar mucho menos (o 0 si todos los nulos eran previos al inicio)
nulos_restantes = df_recortado['cumulative_vaccine_doses_administered'].isnull().sum()
print(f"Cantidad de nulos restantes en vacunas: {nulos_restantes}")

Cantidad de nulos restantes en vacunas: 472


In [104]:
# Calcular cantidad de nulos por país en la columna de vacunas
# Sumar los valores True (nulos) agrupados por nombre de país
nulos_restantes_pais = df_recortado['cumulative_vaccine_doses_administered'].isnull().groupby(df_recortado['country_name']).sum().astype(int)

print("--- NULOS RESTANTES POR PAÍS ---")
display(nulos_restantes_pais)

--- NULOS RESTANTES POR PAÍS ---


country_name
Argentina      0
Brazil         0
Chile         14
Colombia     240
Mexico       217
Peru           1
Name: cumulative_vaccine_doses_administered, dtype: int64

In [105]:
# Lista de claves de país a consultar (Colombia, México, Chile)
claves_seleccionadas = ['CO', 'MX', 'CL']

# Filtrar el dataframe solo para estas claves
# Agrupar por location_key y buscar la fecha máxima
ultimas_fechas = df_recortado[df_recortado['location_key'].isin(claves_seleccionadas)].groupby('location_key')['date'].max()

print("--- ÚLTIMA FECHA REGISTRADA ---")
display(ultimas_fechas)

--- ÚLTIMA FECHA REGISTRADA ---


location_key
CL   2022-09-13
CO   2022-09-13
MX   2022-09-13
Name: date, dtype: datetime64[ns]

In [106]:
# Se observa que los datos faltantes no son al final, sino que se encuentran en el medio
# Rellenar datos mediante interpolación 
col_vacunas = 'cumulative_vaccine_doses_administered'

# Realizar interpolación lineal para rellenar los huecos intermedios
# Agrupar por location_key para respetar la serie de cada país
df_recortado[col_vacunas] = df_recortado.groupby('location_key')[col_vacunas].transform(lambda x: x.interpolate())

# Verificar si quedan nulos pendientes
# Calcular la suma total de nulos en la columna
nulos_finales = df_recortado[col_vacunas].isnull().sum()

print(f"Nulos restantes en vacunas: {nulos_finales}")

# Mostrar una muestra para verificar que ya no hay huecos (opcional)
# Filtrar algunas columnas clave para visualizar
cols_ver = ['date', 'location_key', col_vacunas]

Nulos restantes en vacunas: 0


In [107]:
#CHEQUEO DE NULOS
# Calcular la cantidad de nulos por columna 
nulos_cantidad = df_recortado.isnull().sum()

# Calcular el porcentaje por columna
nulos_porcentaje = (df_recortado.isnull().mean() * 100).round(2)

# Crear el DataFrame de resumen
df_nulos = pd.DataFrame({
    "Cantidad Nulos": nulos_cantidad,
    "Porcentaje (%)": nulos_porcentaje
})

# Filtrar para ver SOLO las columnas que tienen algún nulo
df_nulos_filtrado = df_nulos[df_nulos["Cantidad Nulos"] > 0].sort_values("Porcentaje (%)", ascending=False)

print("--- REPORTE DE VALORES FALTANTES ---")
display(df_nulos_filtrado)

--- REPORTE DE VALORES FALTANTES ---


,Cantidad Nulos,Porcentaje (%)
cumulative_recovered,4156,70.18
new_recovered,3323,56.11
rainfall_mm,113,1.91


        3.5. TRATAMIENTO DE COLUMNA NEW_RECOVERED--> tiene más deñ 50% de datos nulos. 

In [108]:
# Calcular cantidad de nulos por país para la columna new_recovered
# Agrupar por location_key y contar nulos
nulos_cantidad = df_recortado.groupby('location_key')['new_recovered'].apply(lambda x: x.isnull().sum())

# Calcular el porcentaje de nulos por país
# Usar la media de nulos multiplicada por 100
nulos_porcentaje = (df_recortado.groupby('location_key')['new_recovered'].apply(lambda x: x.isnull().mean()) * 100).round(2)

# Crear el DataFrame de resumen
df_nulos_recuperados = pd.DataFrame({
    "Cantidad Nulos": nulos_cantidad,
    "Porcentaje (%)": nulos_porcentaje
})

# Ordenar por porcentaje de mayor a menor para ver los casos más críticos
df_nulos_recuperados = df_nulos_recuperados.sort_values("Porcentaje (%)", ascending=False)

print("--- REPORTE DE VALORES FALTANTES (new_recovered) ---")
display(df_nulos_recuperados)

--- REPORTE DE VALORES FALTANTES (new_recovered) ---


,Cantidad Nulos,Porcentaje (%)
location_key,,
AR,987,100.00
PE,987,100.00
MX,987,100.00
CO,299,30.29
CL,62,6.28
BR,1,0.10


        3.5.1. NEW_RECOVERED DE PAÍSES CON 100% DATOS NULOS--> Ar, Mx y Pe
        Estrategia: Promedio Ponderado con Retraso Temporal.

In [109]:
# Lista de países sin datos confiables de recuperados
paises_sin_datos = ['AR', 'MX', 'PE']

# Configurar los tiempos de recuperación (dias)
dias_leves = 14
dias_severos = 32  # 4.5 semanas redondeado (4.5 * 7 = 31.5)

# Configurar los porcentajes de cada grupo
pct_leves = 0.80
pct_severos = 0.20

# Iterar sobre cada país para aplicar la fórmula ponderada
for pais in paises_sin_datos:
    # Filtrar solo las filas del país actual
    mask = df_recortado['location_key'] == pais
    
    # Calcular recuperados del grupo leve (80% de los casos de hace 14 días)
    recup_leves = df_recortado.loc[mask, 'new_confirmed'].shift(dias_leves) * pct_leves
    
    # Calcular recuperados del grupo severo (20% de los casos de hace 32 días)
    recup_severos = df_recortado.loc[mask, 'new_confirmed'].shift(dias_severos) * pct_severos
    
    # Sumar ambos grupos y restar los fallecidos de hoy
    # Usar fillna(0) para evitar problemas con los primeros días donde el shift genera nulos
    estimado = (recup_leves.fillna(0) + recup_severos.fillna(0)) - df_recortado.loc[mask, 'new_deceased']
    
    # Asignar el resultado a la columna new_recovered
    # Usar clip(lower=0) para evitar números negativos si los fallecidos superan a los recuperados estimados
    df_recortado.loc[mask, 'new_recovered'] = estimado.clip(lower=0)

# Recalcular el acumulado sumando los nuevos recuperados día a día (agrupado por país)
df_recortado['cumulative_recovered'] = df_recortado.groupby('location_key')['new_recovered'].cumsum()

print(f"Recuperados estimados para {paises_sin_datos} usando modelo ponderado (80% en {dias_leves} días, 20% en {dias_severos} días).")

# Verificar resultados mostrando una muestra aleatoria de estos países
cols_ver = ['date', 'location_key', 'new_confirmed', 'new_deceased', 'new_recovered']
display(df_recortado[df_recortado['location_key'].isin(paises_sin_datos)][cols_ver].sample(5))

Recuperados estimados para ['AR', 'MX', 'PE'] usando modelo ponderado (80% en 14 días, 20% en 32 días).


,date,location_key,new_confirmed,new_deceased,new_recovered
5674,2021-12-20,PE,1837.0,40.0,1699.6
202,2020-07-21,AR,5742.0,177.0,3013.6
4866,2022-06-21,MX,16528.0,17.0,4852.8
924,2022-07-13,AR,0.0,0.0,0.0
4926,2022-08-20,MX,1448.0,36.0,9607.0


        3.5.2. NEW_RECOVERED DE PAISES CON ALGÚN % DE DATOS NULOS--> CL y CO
        Estrategia Híbrida

In [110]:
#Respetar el 70% de los datos oficiales y utilizar fórmula de ponderación para el 30% faltante
# Definir lista de países para estrategia híbrida
paises_hibridos = ['CO', 'CL']

# Configurar parámetros del modelo de recuperación
dias_leves = 14
dias_severos = 32
pct_leves = 0.80
pct_severos = 0.20

# Iterar sobre cada país de la lista
for pais in paises_hibridos:
    # Crear filtro para el país actual
    mask = df_recortado['location_key'] == pais
    
    # Calcular la curva estimada teórica basada en contagios pasados
    recup_leves = df_recortado.loc[mask, 'new_confirmed'].shift(dias_leves) * pct_leves
    recup_severos = df_recortado.loc[mask, 'new_confirmed'].shift(dias_severos) * pct_severos
    
    # Sumar grupos y restar fallecidos de hoy
    estimado = (recup_leves.fillna(0) + recup_severos.fillna(0)) - df_recortado.loc[mask, 'new_deceased']
    
    # Asegurar que no existan valores negativos
    estimado = estimado.clip(lower=0)
    
    # Rellenar SOLO los huecos (NaN) usando la estimación
    # Los datos oficiales existentes NO se tocan
    df_recortado.loc[mask, 'new_recovered'] = df_recortado.loc[mask, 'new_recovered'].fillna(estimado)
    
    # Rellenar con 0 los posibles nulos al inicio de la serie (primeras semanas)
    df_recortado.loc[mask, 'new_recovered'] = df_recortado.loc[mask, 'new_recovered'].fillna(0)
    
    # Recalcular la columna acumulada para mantener coherencia matemática
    df_recortado.loc[mask, 'cumulative_recovered'] = df_recortado.loc[mask, 'new_recovered'].cumsum()

print(f"Huecos rellenados exitosamente para: {paises_hibridos}")

# Verificar que ya no queden nulos en estos países
nulos_restantes = df_recortado[df_recortado['location_key'].isin(paises_hibridos)]['new_recovered'].isnull().sum()
print(f"Total de nulos restantes en Colombia y Chile: {nulos_restantes}")

Huecos rellenados exitosamente para: ['CO', 'CL']
Total de nulos restantes en Colombia y Chile: 0


        3.5.2. NEW_RECOVERED DE PAÍS CON 1 SÓLO DATO NULO--> BR
        Interpolación Simple

In [111]:
mask_br = df_recortado['location_key'] == 'BR'

# Rellenar hueco interpolando
df_recortado.loc[mask_br, 'new_recovered'] = df_recortado.loc[mask_br, 'new_recovered'].interpolate()

print("Brasil: Hueco único rellenado por interpolación.")

Brasil: Hueco único rellenado por interpolación.


In [112]:
# Calcular cantidad de nulos por país para la columna new_recovered
# Agrupar por location_key y contar nulos
nulos_cantidad = df_recortado.groupby('location_key')['new_recovered'].apply(lambda x: x.isnull().sum())

# Calcular el porcentaje de nulos por país
# Usar la media de nulos multiplicada por 100
nulos_porcentaje = (df_recortado.groupby('location_key')['new_recovered'].apply(lambda x: x.isnull().mean()) * 100).round(2)

# Crear el DataFrame de resumen
df_nulos_recuperados = pd.DataFrame({
    "Cantidad Nulos": nulos_cantidad,
    "Porcentaje (%)": nulos_porcentaje
})

# Ordenar por porcentaje de mayor a menor para ver los casos más críticos
df_nulos_recuperados = df_nulos_recuperados.sort_values("Porcentaje (%)", ascending=False)

print("--- REPORTE DE VALORES FALTANTES (new_recovered) ---")
display(df_nulos_recuperados)

--- REPORTE DE VALORES FALTANTES (new_recovered) ---


,Cantidad Nulos,Porcentaje (%)
location_key,,
AR,0,0.0
BR,0,0.0
CL,0,0.0
CO,0,0.0
MX,0,0.0
PE,0,0.0


        3.6. TRATAMIENTO DE COLUMNA CUMULATIVE_RECOVERED--> tiene 70.18% de datos nulos. 

In [113]:
# Recalcular la columna acumulada completa para todos los países
# Usar la suma acumulativa (cumsum) de los nuevos recuperados agrupada por país
df_recortado['cumulative_recovered'] = df_recortado.groupby('location_key')['new_recovered'].cumsum()

# Verificar limpieza final
# Contar nulos restantes en la columna acumulada
nulos_finales = df_recortado['cumulative_recovered'].isnull().sum()

print("Columna cumulative_recovered regenerada exitosamente.")
print(f"Total de nulos restantes: {nulos_finales}")

# Mostrar muestra para verificar la coherencia matemática
# Seleccionar un país y ordenar cronológicamente
# Usamos .iloc[100:110] para ver datos intermedios (no puros ceros del inicio)
verificacion_consecutiva = df_recortado[df_recortado['location_key'] == 'CL'] \
    .sort_values('date') \
    .iloc[100:110] 

# Seleccionar columnas clave para la revisión
cols_ver = ['date', 'location_key', 'new_recovered', 'cumulative_recovered']

print("--- VERIFICACIÓN DE SUMA ACUMULATIVA (10 DÍAS CONSECUTIVOS) ---")
display(verificacion_consecutiva[cols_ver])

Columna cumulative_recovered regenerada exitosamente.
Total de nulos restantes: 0
--- VERIFICACIÓN DE SUMA ACUMULATIVA (10 DÍAS CONSECUTIVOS) ---


,date,location_key,new_recovered,cumulative_recovered
2082,2020-04-10,CL,297.0,1571.0
2083,2020-04-11,CL,293.0,1864.0
2084,2020-04-12,CL,195.0,2059.0
2085,2020-04-13,CL,308.0,2367.0
2086,2020-04-14,CL,279.0,2646.0
2087,2020-04-15,CL,291.0,2937.0
2088,2020-04-16,CL,362.0,3299.0
2089,2020-04-17,CL,322.0,3621.0
2090,2020-04-18,CL,414.0,4035.0
2091,2020-04-19,CL,303.0,4338.0


    4. ESTADO ACTUAL DE LOS NULOS POR COLUMNA

In [114]:
# Calcular la cantidad de nulos por columna
nulos_cantidad = df_recortado.isnull().sum()

# Calcular el porcentaje de nulos por columna
nulos_porcentaje = (df_recortado.isnull().mean() * 100).round(2)

# Crear un DataFrame para visualizar mejor los resultados
df_resumen_nulos = pd.DataFrame({
    "Cantidad Nulos": nulos_cantidad,
    "Porcentaje (%)": nulos_porcentaje
})

# Ordenar de mayor a menor porcentaje para identificar prioridades
df_resumen_nulos = df_resumen_nulos.sort_values("Porcentaje (%)", ascending=False)

print("--- ESTADO ACTUAL DE NULOS EN EL DATASET ---")
display(df_resumen_nulos)

--- ESTADO ACTUAL DE NULOS EN EL DATASET ---


,Cantidad Nulos,Porcentaje (%)
rainfall_mm,113,1.91
location_key,0,0.00
country_code,0,0.00
country_name,0,0.00
new_confirmed,0,0.00
new_deceased,0,0.00
cumulative_confirmed,0,0.00
cumulative_deceased,0,0.00
cumulative_vaccine_doses_administered,0,0.00
population,0,0.00


    5. EXPORTAR ARCHIVO FILTRADO

In [115]:
# Definir la ruta completa del archivo de salida
ruta_archivo = r"G:\Mi unidad\Soy Henry\Modulo 4\Proyecto integrador\DatosFinalesFiltrado.csv"

# Exportar el dataframe a la ruta especificada
# Usar index=False para evitar guardar el índice numérico
df_recortado.to_csv(ruta_archivo, index=False)

print("Archivo exportado exitosamente en la ruta indicada.")

Archivo exportado exitosamente en la ruta indicada.
